# Smart Healthcare — Daily Lifestyle Analysis
### Wearable Device Data Analytics

**Department of Artificial Intelligence and Data Science**
KMCT College of Engineering for Emerging Technologies and Management, Perinthalmanna

---

This notebook reproduces the *Smart Healthcare – Daily Lifestyle Analysis* project.
It studies how everyday lifestyle behaviour relates to health indicators recorded by
wearable devices, using **700 daily records from 100 participants observed over 7 days**.

**Workflow**

1. Data loading and inspection
2. Data cleaning and feature preparation
3. Descriptive statistics
4. Group comparison (age group, BMI category, gender)
5. Correlation analysis
6. Visualisation and interpretation

**Dataset attributes (15 columns):** `ID`, `Day`, `Gender`, `Age (years)`,
`Height (meter)`, `Weight (kg)`, `BMI`, `Step Count`, `Distance Travel (Km)`,
`Blood Pressure`, `Heart Rate (BPM)`, `Blood Oxygen Level`,
`Sleep Duration (minutes)`, `Screen Time (minute)`, `Earphone Time (minute)`.

> **Dataset.** This notebook uses the real Kaggle dataset
> **“Smart Healthcare-DailyLife Dataset Wearable device”**
> (`mdimammahdi/smart-healthcare-dailylife-dataset-wearable-device`).
> The cell in Section 2 downloads it automatically with `kagglehub`, so the notebook
> runs end-to-end in Google Colab with no manual upload. If the download is blocked,
> the cell falls back to a direct Kaggle download and finally to a file-upload prompt.

## 1. Import the libraries

`pandas` for tabular data, `NumPy` for numerical work, `Matplotlib` and `Seaborn`
for the charts.

`%matplotlib inline` is what makes every figure appear **inside the notebook** as soon
as the cell runs — without it the plots are drawn to a hidden canvas and nothing shows.

In [ ]:
import subprocess, sys

# kagglehub is used to fetch the dataset; install it if Colab does not have it yet.
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
    import kagglehub

import pandas as pd
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.max_open_warning"] = 0
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

print("pandas", pd.__version__, "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)

## 2. Load the dataset

The data is the public Kaggle dataset
[**Smart Healthcare-DailyLife Dataset Wearable device**](https://www.kaggle.com/datasets/mdimammahdi/smart-healthcare-dailylife-dataset-wearable-device)
- 700 rows (100 participants x 7 days) and 15 columns of wearable-device readings.

The cell below tries several routes in order, so it works in Colab without any setup:

1. A **bundled copy** in the repository (`data/` folder or the working directory) - the
   recommended path when you clone this project from GitHub.
2. `kagglehub.dataset_download(...)` - the normal path for a public Kaggle dataset.
3. A direct download of the Kaggle dataset archive (unzipped locally).
4. A manual upload prompt (`google.colab.files.upload`) as a last resort.

In [ ]:
import os, glob, io, zipfile, urllib.request

KAGGLE_DATASET = "mdimammahdi/smart-healthcare-dailylife-dataset-wearable-device"
DATA_FILE = "smart_healthcare_dailylife_wearable.csv"


def find_csv(root):
    hits = sorted(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True))
    return hits[0] if hits else None


csv_path = None

# 1) bundled copy in the repository (data/ folder or working directory)
for candidate in [os.path.join("data", DATA_FILE), DATA_FILE]:
    if os.path.exists(candidate):
        csv_path = candidate
        break

# 2) kagglehub (public Kaggle dataset, no API key needed)
if csv_path is None:
    try:
        p = kagglehub.dataset_download(KAGGLE_DATASET)
        csv_path = find_csv(p)
        print("Downloaded with kagglehub ->", p)
    except Exception as e:
        print("kagglehub route failed:", type(e).__name__, e)

# 3) direct Kaggle archive download
if csv_path is None:
    try:
        url = f"https://www.kaggle.com/api/v1/datasets/download/{KAGGLE_DATASET}"
        raw = urllib.request.urlopen(url, timeout=120).read()
        zipfile.ZipFile(io.BytesIO(raw)).extractall("kaggle_data")
        csv_path = find_csv("kaggle_data")
        print("Downloaded directly from Kaggle")
    except Exception as e:
        print("direct download failed:", type(e).__name__, e)

# 4) manual upload (Colab only) - last resort
if csv_path is None:
    from google.colab import files
    print("No dataset found. Please upload the CSV file now.")
    uploaded = files.upload()
    csv_path = list(uploaded.keys())[0]

if csv_path is None:
    raise RuntimeError("Could not locate the dataset. "
                       "Place the CSV in the working directory or data/ and re-run.")

print("Using dataset file:", csv_path)

## 3. Load the data into a DataFrame

Read the CSV into a pandas DataFrame and show the first rows.

In [ ]:
df = pd.read_csv(csv_path)
df.head()

## 4. Inspect the dataset

### 4.1 Shape — rows and columns

In [ ]:
df.shape

### 4.2 Column names

In [ ]:
df.columns

### 4.3 Data types and non-null counts

`Gender` and `Blood Pressure` are stored as text (`object`); everything else is numeric.

In [ ]:
df.info()

### 4.4 Missing values

The dataset records **no missing values** in any column.

In [ ]:
df.isnull().sum()

### 4.5 Descriptive statistics

`describe()` gives the count, mean, standard deviation, quartiles and range of every
numeric column.

In [ ]:
df.describe()

### 4.6 Headline averages

In [ ]:
print("Average Age:", df["Age (years)"].mean())
print("Average BMI:", df["BMI"].mean())
print("Average Steps:", df["Step Count"].mean())
print("Average Sleep:", df["Sleep Duration (minutes)"].mean())
print("Average Screen Time:", df["Screen Time (minute)"].mean())
print("Average Heart Rate:", df["Heart Rate (BPM)"].mean())
print("Average Blood Oxygen:", df["Blood Oxygen Level"].mean())

## 5. Data validation — the gender-label problem

The raw file uses **two different codings** for gender: participants 1–28 are coded
`M` / `F`, while participants 29–100 are coded `Male` / `Female`. Plotting sleep by
gender *before* standardising the labels therefore shows **four bars** instead of two,
and the `M` / `F` bars have wider error bars because they rest on fewer participants.

This is the check reported as Figure 10.1 in the project report.

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    x="Gender",
    y="Sleep Duration (minutes)",
    data=df
)

plt.xlabel("Gender")
plt.ylabel("Average Sleep Duration (minutes)")
plt.title("Average Sleep Duration by Gender (before label standardisation)")

plt.tight_layout()
plt.show()

## 6. Data cleaning and feature preparation

1. **Standardise gender labels** — merge `Male`/`Female` into `M`/`F`.
2. **Split blood pressure** — the text `120/82` becomes numeric `SBP` and `DBP`.
3. **Derive age groups** — 18–29, 30–39, 40–49, 50 and above.
4. **Derive BMI categories** — Under (<18.5), Normal (18.5–24.9), Over (25–29.9), Obese (≥30).
5. **Flag short sleep** — sleep below 420 minutes (7 hours).
6. **Classify blood pressure** — Normal, Elevated, Stage 1, Stage 2 (ACC/AHA thresholds).

In [ ]:
# 1. Standardise the gender labels
df["Gender"] = df["Gender"].replace({"Male": "M", "Female": "F"})

# 2. Split the blood-pressure text into numeric systolic / diastolic columns
bp = df["Blood Pressure"].str.split("/", expand=True).astype(int)
df["SBP"], df["DBP"] = bp[0], bp[1]

# 3. Age groups
df["AgeGroup"] = pd.cut(
    df["Age (years)"],
    bins=[17, 29, 39, 49, 60],
    labels=["18-29", "30-39", "40-49", "50+"]
)

# 4. BMI categories
df["BMICategory"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Under", "Normal", "Over", "Obese"],
    right=False
)

# 5. Short-sleep flag (below 7 hours)
df["ShortSleep"] = df["Sleep Duration (minutes)"] < 420

# 6. Blood-pressure category (ACC/AHA)
def bp_category(sbp, dbp):
    if sbp >= 140 or dbp >= 90:
        return "Stage 2"
    if 130 <= sbp < 140 or 80 <= dbp < 90:
        return "Stage 1"
    if 120 <= sbp < 130 and dbp < 80:
        return "Elevated"
    return "Normal"

df["BPCategory"] = [bp_category(s, d) for s, d in zip(df["SBP"], df["DBP"])]

df[["Gender", "Blood Pressure", "SBP", "DBP", "AgeGroup", "BMICategory",
    "ShortSleep", "BPCategory"]].head()

### 6.1 Cleaned summary statistics

The table below is the summary of the 700 records (Table 8.1 of the report).

In [ ]:
summary = df[["Age (years)", "BMI", "Step Count", "Distance Travel (Km)",
              "Heart Rate (BPM)", "Blood Oxygen Level", "Sleep Duration (minutes)",
              "Screen Time (minute)", "Earphone Time (minute)", "SBP", "DBP"]]

summary.describe().T[["mean", "std", "min", "max"]].round(2)

### 6.2 Data-quality observations

- 700 records (100 participants × 7 days), **0 missing values**
- Gender: **57 male, 43 female** participants
- Blood-oxygen readings below 95%: a small group of records
- Heart-rate range: 60–90 BPM

In [ ]:
print("Records:", len(df))
print("Participants:", df["ID"].nunique())
print("Missing values:", int(df.isnull().sum().sum()))
print("Participants by gender:", df.drop_duplicates("ID")["Gender"].value_counts().to_dict())
print("Blood-oxygen readings below 95%:", int((df["Blood Oxygen Level"] < 95).sum()))
print("Heart-rate range:", df["Heart Rate (BPM)"].min(), "to", df["Heart Rate (BPM)"].max(), "BPM")

## 7. Distribution of lifestyle measures

Daily steps, sleep duration and screen time, side by side (Figure 11.1).

- **Steps** are spread widely, from under 1,000 to over 20,000.
- **Sleep** is centred close to seven hours, with a long tail of very short nights.
- **Screen time** is skewed to the right: most days are under six hours, but a few
  days exceed 17 hours.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(df["Step Count"], bins=20, color="#4C72B0", edgecolor="white")
axes[0].set_title("Daily Steps")
axes[0].set_xlabel("Step Count")
axes[0].set_ylabel("Number of Records")

axes[1].hist(df["Sleep Duration (minutes)"], bins=20, color="#55A868", edgecolor="white")
axes[1].set_title("Sleep Duration")
axes[1].set_xlabel("Sleep Duration (minutes)")
axes[1].set_ylabel("Number of Records")

axes[2].hist(df["Screen Time (minute)"], bins=20, color="#C44E52", edgecolor="white")
axes[2].set_title("Screen Time")
axes[2].set_xlabel("Screen Time (minutes)")
axes[2].set_ylabel("Number of Records")

plt.tight_layout()
plt.show()

## 8. Distribution of health indicators

### 8.1 Blood oxygen

Readings range from about 92.7% to 100%. Most records fall between 95% and 100%, and a
small group sits below 95%. The tallest bar is near 100% (Figure 11.2).

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    df["Blood Oxygen Level"],
    bins=10
)

plt.xlabel("Blood Oxygen Level")
plt.ylabel("Number of Records")
plt.title("Blood Oxygen Distribution")

plt.tight_layout()
plt.show()

### 8.2 Heart rate

Heart rate lies between 60 and 90 BPM with a mean of about 73.8 BPM. The most common
range is around 66–69 BPM, and the counts fall gradually towards the higher rates
(Figure 11.3).

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    df["Heart Rate (BPM)"],
    bins=10
)

plt.xlabel("Heart Rate (BPM)")
plt.ylabel("Number of Records")
plt.title("Heart Rate Distribution")

plt.tight_layout()
plt.show()

### 8.3 BMI

BMI ranges from 13.8 to 36.7 with a mean of about 24.7. Most records lie between 18 and
32, with the highest counts near 23–25 (Figure 11.4).

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    df["BMI"],
    bins=10
)

plt.xlabel("BMI")
plt.ylabel("Number of Records")
plt.title("BMI Distribution")

plt.tight_layout()
plt.show()

## 9. Relationship between steps and distance

Step count and distance travelled are almost perfectly linearly related (r ≈ 0.97), as
expected, because distance is derived from the number of steps. A few points lie away
from the main line — for example a record with about 1,000 steps but 7.6 km, one of the
unusual values noted in the report (Figure 11.5).

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(df["Step Count"], df["Distance Travel (Km)"])

plt.xlabel("Step count")
plt.ylabel("Distance Travelled (Km)")
plt.title("Step Count vs Distance Travelled")

plt.tight_layout()
plt.show()

## 10. Screen time and sleep duration

Most records cluster between 100 and 450 minutes of screen time and 350 to 500 minutes
of sleep. Days with very long screen time (over 1,000 minutes) mostly have sleep below
500 minutes. The overall correlation is weak and negative (r ≈ −0.23). A few records
with more than 700 minutes of sleep stand out as outliers (Figure 11.6).

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["Screen Time (minute)"],
    df["Sleep Duration (minutes)"]
)

plt.xlabel("Screen Time (minutes)")
plt.ylabel("Sleep Duration (minutes)")
plt.title("Screen Time vs Sleep Duration")

plt.tight_layout()
plt.show()

## 11. Correlation analysis

Select the numeric columns and compute the pairwise correlation matrix.

In [ ]:
# The 13 original numeric attributes (SBP/DBP are derived, so they are left out
# of the heat-map to match the original project's correlation matrix).
numeric_df = df.drop(columns=["SBP", "DBP"]).select_dtypes(include=np.number)
numeric_df.columns.tolist()

In [ ]:
correlation = numeric_df.corr()

In [ ]:
correlation

### 11.1 Correlation heat-map

- Screen time has a weak negative correlation with steps (r ≈ −0.23) and with sleep
  (r ≈ −0.23).
- Earphone time has the strongest relationship with systolic blood pressure (r ≈ 0.36),
  and steps have a weak negative one (r ≈ −0.19).
- The heat-map also includes the participant `ID`. It correlates with screen time and
  earphone time (r ≈ −0.43 for both) and with steps (r ≈ 0.34). The ID is only an
  identifier, so this reflects how the records of participants 1–28 differ from the rest.

All of these are **association, not proof of cause**, and the sizes are modest
(Figure 11.7).

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.tight_layout()
plt.show()

### 11.2 The two strongest lifestyle relationships

Screen time vs steps, and earphone time vs systolic blood pressure (Figure 11.8).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["Screen Time (minute)"], df["Step Count"], alpha=0.6, color="#4C72B0")
axes[0].set_xlabel("Screen Time (minutes)")
axes[0].set_ylabel("Step Count")
axes[0].set_title("Screen Time vs Steps")

axes[1].scatter(df["Earphone Time (minute)"], df["SBP"], alpha=0.6, color="#C44E52")
axes[1].set_xlabel("Earphone Time (minutes)")
axes[1].set_ylabel("Systolic Blood Pressure (mmHg)")
axes[1].set_title("Earphone Time vs Systolic BP")

plt.tight_layout()
plt.show()

## 12. Comparison by age group

Mean daily steps rise with age, from about 7,600 in the 18–29 group to about 9,400 in
the 50-and-above group. Screen time is highest in the 30–39 group and lowest in the
50-and-above group (Figure 11.9).

In [ ]:
age_stats = df.groupby("AgeGroup", observed=True)[["Step Count", "Screen Time (minute)"]].mean()
print(age_stats.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_stats["Step Count"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Mean Daily Steps by Age Group")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("Mean Steps")
axes[0].tick_params(axis="x", rotation=0)

age_stats["Screen Time (minute)"].plot(kind="bar", ax=axes[1], color="#DD8452")
axes[1].set_title("Mean Screen Time by Age Group")
axes[1].set_xlabel("Age Group")
axes[1].set_ylabel("Mean Screen Time (minutes)")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 13. Comparison by BMI category

The obese group has the lowest mean step count and the normal-weight group the highest.
The under-weight group has the shortest mean sleep. The BMI groups are of different
sizes, so these differences should be read with care (Figure 11.10).

In [ ]:
bmi_stats = df.groupby("BMICategory", observed=True)[["Step Count", "Sleep Duration (minutes)"]].mean()
print(bmi_stats.round(1))
print("\nRecords per BMI category:")
print(df["BMICategory"].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bmi_stats["Step Count"].plot(kind="bar", ax=axes[0], color="#55A868")
axes[0].set_title("Mean Daily Steps by BMI Category")
axes[0].set_xlabel("BMI Category")
axes[0].set_ylabel("Mean Steps")
axes[0].tick_params(axis="x", rotation=0)

bmi_stats["Sleep Duration (minutes)"].plot(kind="bar", ax=axes[1], color="#8172B2")
axes[1].set_title("Mean Sleep by BMI Category")
axes[1].set_xlabel("BMI Category")
axes[1].set_ylabel("Mean Sleep (minutes)")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 14. Comparison by gender

The differences between male and female participants are small for every measure
(Table 11.2).

In [ ]:
gender_stats = df.groupby("Gender")[[
    "Step Count", "Sleep Duration (minutes)", "Screen Time (minute)", "Heart Rate (BPM)"
]].mean().round(1)

gender_stats.columns = ["Mean steps per day", "Mean sleep (minutes)",
                        "Mean screen time (minutes)", "Mean heart rate (BPM)"]
gender_stats

## 15. Blood-pressure categories

Grouping the single readings with the ACC/AHA thresholds gives the distribution below
(Table 11.3). These categories are a screening-style grouping of single readings — a
diagnosis of high blood pressure needs repeated measurement by a medical professional.

In [ ]:
bp_counts = df["BPCategory"].value_counts().reindex(["Normal", "Elevated", "Stage 1", "Stage 2"])
bp_table = pd.DataFrame({
    "Records": bp_counts,
    "Share (%)": (bp_counts / len(df) * 100).round(1)
})
print(bp_table)

plt.figure(figsize=(8, 5))
bp_counts.plot(kind="bar", color=["#55A868", "#DD8452", "#C44E52", "#8C1D1D"])
plt.title("Blood-Pressure Categories (ACC/AHA)")
plt.xlabel("Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 16. Trend across the observed days

Mean steps are highest on day 1 and lowest on day 6. The dataset does not state the
calendar day of each observation, so the pattern cannot be tied to weekdays or weekends
(Figure 11.11).

In [ ]:
day_stats = df.groupby("Day")["Step Count"].mean()
print(day_stats.round(1))

plt.figure(figsize=(8, 5))
plt.plot(day_stats.index, day_stats.values, marker="o", color="#4C72B0")
plt.xlabel("Day")
plt.ylabel("Mean Steps")
plt.title("Mean Steps Across the Seven Observed Days")
plt.xticks(day_stats.index)

plt.tight_layout()
plt.show()

## 17. Overview of findings

| Area | Finding |
|---|---|
| **Activity** | Mean ≈ 8,357 steps and 5.89 km per day. About 38% of days reached 10,000 steps and about 24% were below 5,000. |
| **Sleep** | Mean ≈ 420 minutes (7.0 hours). About 49% of nights were shorter than 7 hours. |
| **Screen and earphone time** | Mean screen time ≈ 326 minutes and earphone time ≈ 130 minutes per day. |
| **Vitals** | Mean heart rate ≈ 73.8 BPM, blood oxygen ≈ 97.5%, blood pressure ≈ 120/77 mmHg. |

In [ ]:
print("Mean steps:", round(df["Step Count"].mean()))
print("Median steps:", round(df["Step Count"].median()))
print("Mean distance (km):", round(df["Distance Travel (Km)"].mean(), 2))
print("Days >= 10,000 steps: {:.1f}%".format((df["Step Count"] >= 10000).mean() * 100))
print("Days < 5,000 steps: {:.1f}%".format((df["Step Count"] < 5000).mean() * 100))
print("Mean sleep (min):", round(df["Sleep Duration (minutes)"].mean(), 1))
print("Nights < 7 hours: {:.1f}%".format((df["Sleep Duration (minutes)"] < 420).mean() * 100))
print("Mean screen time (min):", round(df["Screen Time (minute)"].mean(), 1))
print("Mean earphone time (min):", round(df["Earphone Time (minute)"].mean(), 1))
print("Mean heart rate (BPM):", round(df["Heart Rate (BPM)"].mean(), 1))
print("Mean blood oxygen (%):", round(df["Blood Oxygen Level"].mean(), 2))
print("Mean blood pressure:", round(df["SBP"].mean(), 1), "/", round(df["DBP"].mean(), 1), "mmHg")

## 18. Discussion

- About half of the nights (≈49%) were shorter than the seven-hour reference, which
  suggests that short sleep is common in the group.
- Days with more screen time tend to have fewer steps and less sleep, although the
  relationship is weak.
- Earphone time shows the strongest link to systolic blood pressure in this dataset.
  Because other factors such as age and fitness are not controlled, this should be
  tested further before any conclusion is drawn.
- Older participants walked more on average, which is the opposite of what might be
  assumed.
- Findings are limited by the short observation period and the data-quality notes below.

### Limitations

- The dataset covers only 7 days for each of 100 participants, so long-term trends
  cannot be studied.
- The analysis is descriptive. Correlations show **association, not cause**.
- Participants 1–28 show less day-to-day variation than participants 29–100 (for
  example a constant heart rate and an identical blood-pressure reading across several
  days). The records may have been collected or entered differently and should be
  verified with the data provider.
- A few values look unusual and were kept as provided — for example a BMI of 13.8 for
  participant 8 and 1,054 steps against 7.62 km on day 2 of the same participant.
- No predictive model was built in this report.
- The dataset does not state how the devices were worn or calibrated.

### Future enhancements

- Predictive models for blood-pressure category or sleep quality.
- Longer observation periods, with a larger and more varied group of participants.
- An interactive dashboard for browsing the results.
- Personalised alerts, for example for short sleep or very low activity.
- Automated data-quality checks for outliers and repeated values.
- Integration with live data from a wearable device.

## 19. Conclusion

*Smart Healthcare – Daily Lifestyle Analysis* studied how everyday lifestyle behaviour
relates to health indicators recorded by wearable devices, using 700 daily records from
100 participants.

The project loaded and cleaned the dataset, derived age, BMI and blood-pressure groups,
and carried out descriptive, comparative and correlation analysis in Python. The results
were presented as tables and charts together with a discussion of their meaning and
limits.

**Achievements**

- Prepared a clean, analysis-ready dataset from the wearable-device records.
- Summarised activity, sleep, screen time and vital signs of the participants.
- Compared lifestyle measures across age groups, gender and BMI categories.
- Measured correlations between lifestyle measures and vital signs.
- Produced clear charts and tables for every finding.

Careful analysis of wearable data can turn raw readings into useful insight about daily
health habits.

---

### References

1. McKinney, W. *Python for Data Analysis*, 3rd ed., O'Reilly Media, 2022.
2. VanderPlas, J. *Python Data Science Handbook*, 2nd ed., O'Reilly Media, 2022.
3. Han, J., Kamber, M., Pei, J. *Data Mining: Concepts and Techniques*, 3rd ed., Morgan Kaufmann, 2012.
4. Piwek, L., Ellis, D. A., Andrews, S., Joinson, A. "The Rise of Consumer Health Wearables: Promises and Barriers", *PLoS Medicine*, 13(2), e1001953, 2016.
5. World Health Organization. *Physical activity* (fact sheet).
6. Hirshkowitz, M., et al. "National Sleep Foundation's sleep time duration recommendations", *Sleep Health*, 1(1), 40–43, 2015.
7. Whelton, P. K., et al. "2017 ACC/AHA Guideline for the Prevention, Detection, Evaluation, and Management of High Blood Pressure in Adults", *Hypertension*, 71(6), e13–e115, 2018.
8. pandas documentation — https://pandas.pydata.org/docs/
9. NumPy documentation — https://numpy.org/doc/
10. Matplotlib documentation — https://matplotlib.org/stable/
11. Kaggle dataset — https://www.kaggle.com/datasets/mdimammahdi/smart-healthcare-dailylife-dataset-wearable-device